# 6-4. ARIMA·GARCH·SV 자체 커버리지 vs 기존 리샘플링

모델 자체의 확률적 신뢰구간을 직접 써서 기존 복원추출 커버리지와 비교한다.

- **ARIMA**: `predict(return_conf_int=True)` -- 상태공간 이론구간
- **GARCH(1,1)**: ARIMA 잔차에 GARCH 적합, h스텝 분산 직접예측
- **SV(QML)**: 로그변동성을 잠재상태로 칼만필터 추정

대표변수 `X3_l1`, 점예측은 전부 Auto-ARIMA로 통일(구간 방식만 비교).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pmdarima as pm
from arch import arch_model
from scipy import stats

H = 4
INITIAL_CALIB = 15
MIN_TRAIN = 20
TARGET_VAR = "X3_l1"
ALPHA = 0.10
Z = stats.norm.ppf(1 - ALPHA / 2)  # 90% 구간 -> z=1.645

raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
vals = raw[["quarter", TARGET_VAR]].dropna()[TARGET_VAR].values.astype(float)
n = len(vals)
origins = list(range(MIN_TRAIN, n - H + 1))
scored_origins = origins[INITIAL_CALIB:]
print(f"대표변수: X3 ({TARGET_VAR}), 채점 origin {len(scored_origins)}개, z(90%)={Z:.3f}")


대표변수: X3 (X3_l1), 채점 origin 22개, z(90%)=1.645


## 1. 세 가지 "자체 커버리지" 구성 함수

셋 다 같은 ARIMA 점예측을 공유하고(`m.predict(n_periods=H)`), 구간을 만드는 방식만 다르다.

**동기화 메모**: 아래 `fit_arima`/`arima_native_interval`/`garch_native_interval`/`sv_qml_interval` 함수들은 `6-5` 노트북에도 거의 동일하게 복사돼 있다(공용 모듈로 분리하지 않기로 함 -- 노트북은 한 파일만 읽어도 이해되는 게 더 중요하다고 판단). **여기를 고치면 6-5도 확인할 것.**


In [2]:
def fit_arima(train):
    return pm.auto_arima(train, seasonal=False, suppress_warnings=True, error_action="ignore", information_criterion="aic")


def arima_native_interval(m, h=H, alpha=ALPHA):
    fc, conf = m.predict(n_periods=h, return_conf_int=True, alpha=alpha)
    return np.asarray(fc), conf[:, 0], conf[:, 1]


def garch_native_interval(m, train, h=H, alpha=ALPHA):
    fc = np.asarray(m.predict(n_periods=h))
    insample_resid = train - m.predict_in_sample()
    try:
        am = arch_model(insample_resid, mean="Zero", vol="Garch", p=1, q=1, rescale=False)
        res = am.fit(disp="off")
        variance_h = res.forecast(horizon=h, reindex=False).variance.values[-1]
        sigma_h = np.sqrt(variance_h)
    except Exception:
        sigma_h = np.repeat(insample_resid.std(), h)
    lo, hi = fc - Z * sigma_h, fc + Z * sigma_h
    return fc, lo, hi


## 1-1. SV(QML) -- Harvey·Ruiz·Shephard(1994) 방법

- 상태: `h_t = phi*h_(t-1) + eta_t`
- 관측: `log(잔차^2) = h_t + log(e_t^2)`

`log(e_t^2)`는 평균(-1.2704)·분산(pi^2/2)이 알려진 값이라 정규근사로 칼만필터를 쓸 수 있다(QML, Quasi-Maximum-Likelihood).


In [3]:
from scipy.optimize import minimize

XI_MEAN, XI_VAR = -1.2704, np.pi ** 2 / 2  # log(chi-sq_1)의 평균/분산 (알려진 상수)
MAX_PHI = 0.97  # phi가 1에 가까워지면 다단계 분산이 exp()를 거치며 폭발하는 것을 막는 하드 상한


def _kalman_negloglik(params, z_centered):
    phi = MAX_PHI * np.tanh(params[0])
    sigma_eta2 = np.exp(params[1]) ** 2
    P0 = sigma_eta2 / (1 - phi ** 2) if abs(phi) < 0.999 else sigma_eta2 * 100
    h_pred, P_pred, loglik = 0.0, P0, 0.0
    for z in z_centered:
        F = P_pred + XI_VAR
        v = z - h_pred
        loglik += -0.5 * (np.log(2 * np.pi * F) + v ** 2 / F)
        K = P_pred / F
        h_filt, P_filt = h_pred + K * v, P_pred * (1 - K)
        h_pred, P_pred = phi * h_filt, phi ** 2 * P_filt + sigma_eta2
    return -loglik


def fit_sv_qml(resid):
    z = np.log(resid ** 2 + 1e-8)
    z_mean = z.mean()
    z_centered = z - z_mean
    res = minimize(_kalman_negloglik, x0=[1.5, np.log(0.3)], args=(z_centered,), method="Nelder-Mead")
    phi = MAX_PHI * np.tanh(res.x[0])
    sigma_eta2 = np.exp(res.x[1]) ** 2

    P0 = sigma_eta2 / (1 - phi ** 2) if abs(phi) < 0.999 else sigma_eta2 * 100
    h_pred, P_pred = 0.0, P0
    for z_t in z_centered:
        F = P_pred + XI_VAR
        v = z_t - h_pred
        K = P_pred / F
        h_filt, P_filt = h_pred + K * v, P_pred * (1 - K)
        h_pred, P_pred = phi * h_filt, phi ** 2 * P_filt + sigma_eta2
    return phi, sigma_eta2, h_filt, P_filt, z_mean


def sv_qml_interval(m, train, h=H, alpha=ALPHA):
    fc = np.asarray(m.predict(n_periods=h))
    insample_resid = train - m.predict_in_sample()
    try:
        phi, sigma_eta2, h_filt, P_filt, z_mean = fit_sv_qml(insample_resid)
        variances = []
        for k in range(1, h + 1):
            h_mean_k = phi ** k * h_filt
            h_var_k = phi ** (2 * k) * P_filt + sigma_eta2 * sum(phi ** (2 * j) for j in range(k))
            log_var_level = z_mean - XI_MEAN + h_mean_k
            variances.append(np.exp(log_var_level + h_var_k / 2))
        sigma_h = np.sqrt(np.array(variances))
    except Exception:
        sigma_h = np.repeat(insample_resid.std(), h)
    lo, hi = fc - Z * sigma_h, fc + Z * sigma_h
    return fc, lo, hi


## 2. 기존 복원추출 방식(비교 기준)

같은 origin·같은 ARIMA 점예측 위에, 지금까지 써온 방식대로 과거오차(h별)를 복원추출해서 구간을 만든다.


In [4]:
def rng_for(name):
    return np.random.default_rng(int.from_bytes(name.encode(), "little") % (2**32))


def resample_interval(fc, calib_resid_by_h, rng, h=H, alpha=ALPHA):
    lo, hi = [], []
    for hh in range(1, h + 1):
        samples = fc[hh - 1] + rng.choice(calib_resid_by_h[hh], size=500, replace=True)
        l, u = np.quantile(samples, [alpha / 2, 1 - alpha / 2])
        lo.append(l); hi.append(u)
    return np.array(lo), np.array(hi)


## 2-1. 모델별 자체 커버리지 계산 방법 요약

공통: `구간 = 점예측 ± 1.645*sigma_h`, sigma_h만 다르게 구한다(리샘플링 없음).

| 모델 | sigma_h 구하는 법 |
|---|---|
| ARIMA | 상태공간 이론공식(라이브러리 자동, `return_conf_int=True`) |
| GARCH(1,1) | ARIMA 잔차에 GARCH 적합 -> `forecast(horizon=h)`로 h스텝 분산 직접예측 |
| SV(QML) | `log(잔차^2)`를 관측치로 칼만필터 -> h_t 추정 -> h스텝 예측 -> exp() |


## 3. Walk-forward 비교 -- origin마다 4가지 방식(기존 리샘플링 + ARIMA/GARCH/SV(QML) 자체) 전부 계산

**캐싱**: 37개 origin마다 ARIMA+GARCH+SV(QML)를 새로 학습시키는 비용이 커서, 결과를 `cache_73_result.pkl`에 저장해두고 다음에 노트북을 고칠 때는 캐시를 읽어서 재계산을 건너뛴다.


In [5]:
import os
import pickle

CACHE_PATH = "../cache/cache_73_result.pkl"
methods = ["기존(리샘플링)", "ARIMA 자체", "GARCH(1,1) 자체", "SV(QML) 자체"]

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "rb") as f:
        hit_df, width_df, store = pickle.load(f)
    print(f"캐시에서 로드: hit_df {len(hit_df)}행 (재계산 스킵)")
else:
    rng = rng_for("arima_native_compare")
    hit_rows = []      # 방식 x h 별 적중 여부
    width_rows = []    # 방식 x h 별 구간폭
    calib_pool = {h: np.array([]) for h in range(1, H + 1)}  # 확장되는 과거오차 풀(리샘플링용)
    store = {name: {"fc": [], "lo1": [], "hi1": [], "actual": []} for name in methods}  # scale 스윕용 원본 저장(7번 섹션)

    for oi, origin in enumerate(origins):
        train = vals[:origin]
        actual = vals[origin:origin + H]
        m = fit_arima(train)
        fc_native = np.asarray(m.predict(n_periods=H))

        if oi >= INITIAL_CALIB:
            # (A) 기존 리샘플링
            lo_r, hi_r = resample_interval(fc_native, calib_pool, rng)
            # (B) ARIMA 자체
            _, lo_a, hi_a = arima_native_interval(m)
            # (C) GARCH(1,1) 자체
            _, lo_g, hi_g = garch_native_interval(m, train)
            # (D) SV(QML) 자체
            _, lo_q, hi_q = sv_qml_interval(m, train)

            for name, lo_h_arr, hi_h_arr in [
                ("기존(리샘플링)", lo_r, hi_r),
                ("ARIMA 자체", lo_a, hi_a),
                ("GARCH(1,1) 자체", lo_g, hi_g),
                ("SV(QML) 자체", lo_q, hi_q),
            ]:
                store[name]["fc"].append(fc_native)
                store[name]["lo1"].append(lo_h_arr)
                store[name]["hi1"].append(hi_h_arr)
                store[name]["actual"].append(actual)
                for hh in range(1, H + 1):
                    y = actual[hh - 1]
                    lo_h, hi_h = lo_h_arr[hh - 1], hi_h_arr[hh - 1]
                    hit_rows.append({"origin": origin, "h": hh, "방식": name, "적중": lo_h <= y <= hi_h})
                    width_rows.append({"origin": origin, "h": hh, "방식": name, "폭": hi_h - lo_h})

        # 다음 origin을 위해 과거오차 풀 갱신(기존 리샘플링용, h=1..4 각각의 실제 오차 축적)
        for hh in range(1, H + 1):
            calib_pool[hh] = np.append(calib_pool[hh], actual[hh - 1] - fc_native[hh - 1])

    hit_df = pd.DataFrame(hit_rows)
    width_df = pd.DataFrame(width_rows)
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    with open(CACHE_PATH, "wb") as f:
        pickle.dump((hit_df, width_df, store), f)
    print(f"새로 계산 후 캐시 저장: hit_df {len(hit_df)}행")

print(f"완료: {hit_df['origin'].nunique()}개 origin x {H}개 h x {len(methods)}개 방식")


캐시에서 로드: hit_df 352행 (재계산 스킵)
완료: 22개 origin x 4개 h x 4개 방식


## 4. 결과 -- 방식별 커버리지(%) vs 평균 구간폭


In [6]:
summary = hit_df.groupby("방식")["적중"].mean().mul(100).round(1).rename("커버리지(%)")
width_summary = width_df.groupby("방식")["폭"].mean().round(2).rename("평균 구간폭")
result_df = pd.concat([summary, width_summary], axis=1).loc[methods]
result_df["목표(90%)와 차이"] = (result_df["커버리지(%)"] - 90).round(1)
display(result_df)


,커버리지(%),평균 구간폭,목표(90%)와 차이
방식,,,
기존(리샘플링),73.9,600.29,-16.1
ARIMA 자체,51.1,404.84,-38.9
"GARCH(1,1) 자체",36.4,275.93,-53.6
SV(QML) 자체,42.0,281.75,-48.0


## 5. h별 세부 비교 (h가 커질수록 자체 커버리지가 얼마나 벌어지는지)


In [7]:
pivot_cov = hit_df.groupby(["h", "방식"])["적중"].mean().mul(100).round(1).unstack()[methods]
pivot_width = width_df.groupby(["h", "방식"])["폭"].mean().round(2).unstack()[methods]
print("h별 커버리지(%):")
display(pivot_cov)
print("h별 평균 구간폭:")
display(pivot_width)


h별 커버리지(%):


방식,기존(리샘플링),ARIMA 자체,"GARCH(1,1) 자체",SV(QML) 자체
h,,,,
1,72.7,54.5,63.6,68.2
2,72.7,54.5,40.9,50.0
3,72.7,45.5,18.2,22.7
4,77.3,50.0,22.7,27.3


h별 평균 구간폭:


방식,기존(리샘플링),ARIMA 자체,"GARCH(1,1) 자체",SV(QML) 자체
h,,,,
1,309.28,211.60,275.35,282.78
2,530.95,360.47,275.59,283.00
3,666.12,479.67,276.06,282.02
4,894.79,567.63,276.71,279.19


## 6. 결론 -- 모델 자체 커버리지가 기존 리샘플링보다 낫지 않다

세 방식 다 리샘플링(73.9%)보다 목표(90%)에서 더 멀다 -- ARIMA 51.1%, GARCH 36.4%, SV(QML) 42.0%. 이 수치 자체가 반증 근거다.

**원인 1**: 정규분포 가정이 이 데이터에 안 맞음(5/6 변수 정규성 기각, 3-2 확인). 리샘플링은 실제 분포를 그대로 써서 이 문제가 없다.

**원인 2**: h가 커질수록 GARCH/SV 커버리지가 무너진다(h=1->4: GARCH 63.6%->22.7%, SV 68.2%->27.3%, 리샘플링은 72~77%로 유지). 구간폭이 h에 따라 안 넓어지기 때문(GARCH/SV는 275->277, 리샘플링/ARIMA는 2.7~2.9배 증가) -- "변동성 평균회귀" 가정 탓.

**종합(8번 답)**: 셋 다 기존 리샘플링보다 못하다(비정규성 + 평균회귀 가정 부적합). 기존 방식 유지.

## 7. 참고 -- scale로 90% 맞춘 뒤 폭 비교 (보조 검증)

scale로 커버리지를 억지로 90%에 맞추고 그 수치 자체를 성과로 쓰면 순환논리다(3-1에서 실제로 있었던 버그). 여기선 "다 90%로 맞춰놓고 그때 필요한 폭"만 참고로 본다 -- 폭은 튜닝 목표가 아니라 결과로 나오는 값이라 비교에 유효하다(ROC에서 위양성률 고정 후 재현율 비교하는 논리와 같음). 6번 결론의 근거는 아니고 보조 자료.

In [8]:
def coverage_width_at_scale(name, scale):
    fcs = np.array(store[name]["fc"])
    lo1 = np.array(store[name]["lo1"])
    hi1 = np.array(store[name]["hi1"])
    act = np.array(store[name]["actual"])
    lo_s = fcs - scale * (fcs - lo1)
    hi_s = fcs + scale * (hi1 - fcs)
    hits = (lo_s <= act) & (act <= hi_s)
    widths = hi_s - lo_s
    return hits.mean() * 100, widths.mean()


scale_grid = np.arange(0.3, 6.01, 0.02)
tuned_rows = []
for name in methods:
    best_scale, best_diff, best_cov, best_width = 1.0, 1e9, None, None
    for s in scale_grid:
        cov, wid = coverage_width_at_scale(name, s)
        diff = abs(cov - 90)
        if diff < best_diff:
            best_diff, best_scale, best_cov, best_width = diff, s, cov, wid
    tuned_rows.append({
        "방식": name, "튜닝된 scale": round(best_scale, 2),
        "튜닝후 커버리지(%)": round(best_cov, 1), "튜닝후 평균폭": round(best_width, 2),
    })
tuned_df = pd.DataFrame(tuned_rows).set_index("방식")
display(tuned_df)


,튜닝된 scale,튜닝후 커버리지(%),튜닝후 평균폭
방식,,,
기존(리샘플링),1.58,89.8,948.45
ARIMA 자체,2.46,89.8,995.91
"GARCH(1,1) 자체",4.04,89.8,1114.75
SV(QML) 자체,3.50,89.8,986.12


## 부록 -- 이 숫자가 버그 아닌지 검증 (시뮬레이션)

낮은 값이 버그인지 확인하기 위해, 실제 코드(`arima_native_interval`/`garch_native_interval`/`sv_qml_interval`)를 그대로 **정규분포/SV 구조를 정확히 따르는 깨끗한 시뮬레이션 데이터**에 돌려본다. 코드가 정상이면 이상적 데이터에서는 목표(90%)에 훨씬 가까운 값이 나와야 한다.


In [9]:
sim_rng = np.random.default_rng(1)


def simulate_ar1(n_total, phi_true=0.6, noise_std=10.0, rng=sim_rng):
    y = [0.0]
    for _ in range(n_total - 1):
        y.append(phi_true * y[-1] + rng.normal(0, noise_std))
    return np.array(y)


# (1) ARIMA/GARCH 검증 -- 이상적인 정규분포 AR(1) 데이터(구조변화·비정규성 없음)
hits_arima_sim, hits_garch_sim = [], []
for trial in range(30):
    y_sim = simulate_ar1(60)
    for origin_sim in [35, 45, 50]:
        train_sim, actual_sim = y_sim[:origin_sim], y_sim[origin_sim:origin_sim + H]
        m_sim = fit_arima(train_sim)
        _, lo_a_sim, hi_a_sim = arima_native_interval(m_sim)
        _, lo_g_sim, hi_g_sim = garch_native_interval(m_sim, train_sim)
        for hh in range(H):
            hits_arima_sim.append(lo_a_sim[hh] <= actual_sim[hh] <= hi_a_sim[hh])
            hits_garch_sim.append(lo_g_sim[hh] <= actual_sim[hh] <= hi_g_sim[hh])

cov_arima_sim = np.mean(hits_arima_sim) * 100
cov_garch_sim = np.mean(hits_garch_sim) * 100
print(f"[시뮬레이션] ARIMA 자체 커버리지:      {cov_arima_sim:.1f}%  (실제데이터: 51.1%)")
print(f"[시뮬레이션] GARCH(1,1) 자체 커버리지: {cov_garch_sim:.1f}%  (실제데이터: 36.4%)")


[시뮬레이션] ARIMA 자체 커버리지:      80.3%  (실제데이터: 51.1%)
[시뮬레이션] GARCH(1,1) 자체 커버리지: 75.6%  (실제데이터: 36.4%)


In [10]:
def simulate_sv(n_total, h_horizon, phi_true=0.9, sigma_eta_true=0.3, rng=sim_rng):
    h_true = [0.0]
    for _ in range(n_total - 1 + h_horizon):
        h_true.append(phi_true * h_true[-1] + rng.normal(0, sigma_eta_true))
    h_true = np.array(h_true)
    sigma_true = np.exp(h_true / 2) * 10
    e = rng.normal(0, 1, size=len(h_true))
    return sigma_true * e


# (2) SV(QML) 검증 -- 진짜 SV(확률변동성) 프로세스로 시뮬레이션
hits_sv_sim = []
for trial in range(50):
    y_sim = simulate_sv(60, H)
    for origin_sim in [40, 50]:
        train_sim, actual_sim = y_sim[:origin_sim], y_sim[origin_sim:origin_sim + H]
        m_sim = fit_arima(train_sim)
        _, lo_q_sim, hi_q_sim = sv_qml_interval(m_sim, train_sim)
        for hh in range(H):
            hits_sv_sim.append(lo_q_sim[hh] <= actual_sim[hh] <= hi_q_sim[hh])

cov_sv_sim = np.mean(hits_sv_sim) * 100
print(f"[시뮬레이션] SV(QML) 자체 커버리지: {cov_sv_sim:.1f}%  (실제데이터: 42.0%)")


[시뮬레이션] SV(QML) 자체 커버리지: 88.5%  (실제데이터: 42.0%)


이상적인 데이터에서는 전부 실제데이터보다 목표(90%)에 훨씬 가까운 값이 나온다 -- 코드 자체는 정상이고, 실제데이터에서 크게 나빠지는 건 비정규성·구조변화 같은 데이터 특성 때문이라는 걸 재확인한다.
